# Subteam A — YOLO Model Training
**Dataset:** Fruit Detection — Roboflow (fruit-detection-2aajq/fruit-nxoac)  

**Dataset Info:**
- Total images: 825
- Classes (6): Apple, Tomato, Cauliflower, Adrak, Bitter Gourd, Capsicum
- Source: https://universe.roboflow.com/fruit-detection-2aajq/fruit-nxoac

In [ ]:
!pip install ultralytics roboflow

In [ ]:
from __future__ import print_function
from ultralytics import YOLO
from roboflow import Roboflow
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from IPython.display import Image, display

print('Libraries loaded!')
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## Step 2 - Download Dataset from Roboflow
Set ROBOFLOW_API_KEY in your local environment using your Roboflow key. Never paste a private key into a committed notebook.

In [ ]:
api_key = os.environ.get("ROBOFLOW_API_KEY")
if not api_key:
    raise ValueError("Set ROBOFLOW_API_KEY in your environment before downloading the dataset.")
rf = Roboflow(api_key=api_key)
project = rf.workspace("fruit-detection-2aajq").project("fruit-nxoac")
dataset = project.version(4).download("yolov8")

print('Dataset downloaded to:', dataset.location)
for item in os.listdir(dataset.location):
    print(' -', item)

In [ ]:
trainPath = os.path.join(dataset.location, 'train', 'images')
valPath   = os.path.join(dataset.location, 'valid', 'images')
testPath  = os.path.join(dataset.location, 'test',  'images')

trainCount = len(os.listdir(trainPath))
valCount   = len(os.listdir(valPath))
testCount  = len(os.listdir(testPath))

print('Train images:', trainCount)
print('Val images:  ', valCount)
print('Test images: ', testCount)
print('Total:       ', trainCount + valCount + testCount)

## Step 3 - Create data.yaml
Same format as Lab 7. Tells YOLO where the images are and what the 6 classes are.

In [ ]:
# Use the data.yaml that Roboflow automatically generated during download
yamlPath = os.path.join(dataset.location, "data.yaml")

print("Using Roboflow data.yaml at:", yamlPath)
print("\nContents:")
with open(yamlPath, "r") as f:
    print(f.read())

## Pipeline 1 - Train YOLOv5
- epochs = 50
- imgsz = 640
- batch = 16
- patience = 10 (early stopping)

In [ ]:
modelV5 = YOLO('yolov5n.pt')
print('YOLOv5n loaded!')

In [ ]:
resultsV5 = modelV5.train(
    data     = yamlPath,
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    name     = 'fruit_yolov5',
    patience = 10
)

print('YOLOv5 training complete!')
print('Results saved to: runs/detect/fruit_yolov5/')

In [ ]:
bestPathV5 = 'runs/detect/fruit_yolov5/weights/best.pt'
modelV5Best = YOLO(bestPathV5)
print('Best YOLOv5 weights loaded!')

## Pipeline 2 - Train YOLOv8
- Same epochs and imgsz as YOLOv5 for fair comparison
- batch = 8 becuse yolov8 uses more GPU memory

In [ ]:
modelV8 = YOLO('yolov8n.pt')
print('YOLOv8n loaded!')

In [ ]:
resultsV8 = modelV8.train(
    data     = yamlPath,
    epochs   = 50,
    imgsz    = 640,
    batch    = 8,
    name     = 'fruit_yolov8',
    patience = 10
)

print('YOLOv8 training complete!')
print('Results saved to: runs/detect/fruit_yolov8/')

In [ ]:
bestPathV8 = 'runs/detect/fruit_yolov8/weights/best.pt'
modelV8Best = YOLO(bestPathV8)
print('Best YOLOv8 weights loaded!')

## Step 4 - Predictions on Test Images

In [ ]:
testImages = [
    os.path.join(testPath, f)
    for f in os.listdir(testPath)
    if f.endswith(('.jpg', '.jpeg', '.png'))
]
sampleImages = testImages[:3]

print('Test images found:', len(testImages))
print('Using first 3 for display:')
for img in sampleImages:
    print(' -', os.path.basename(img))

In [ ]:
#yolov5 predictions
print('YOLOv5 Predictions:')
resultsV5Pred = modelV5Best(sampleImages, iou=0.5)
for i, result in enumerate(resultsV5Pred):
    result.save(filename=f'pred_v5_{i}.jpg')
    display(Image(f'pred_v5_{i}.jpg'))
    print(f'Image {i}:', result.verbose())

In [ ]:
# yolov8 predictions
print('YOLOv8 Predictions:')
resultsV8Pred = modelV8Best(sampleImages, iou=0.5)
for i, result in enumerate(resultsV8Pred):
    result.save(filename=f'pred_v8_{i}.jpg')
    display(Image(f'pred_v8_{i}.jpg'))
    print(f'Image {i}:', result.verbose())

## Step 5 - Side-by-Side Comparison

In [ ]:
testImage = cv2.imread(sampleImages[0])

predV5 = modelV5Best(testImage)
predV8 = modelV8Best(testImage)

# results[0].plot() draws bounding boxes
annotatedV5 = cv2.cvtColor(predV5[0].plot(), cv2.COLOR_BGR2RGB)
annotatedV8 = cv2.cvtColor(predV8[0].plot(), cv2.COLOR_BGR2RGB)
originalRGB = cv2.cvtColor(testImage,        cv2.COLOR_BGR2RGB)

plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(originalRGB)
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(annotatedV5)
plt.title('YOLOv5 Predictions\n(Pipeline 1)')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(annotatedV8)
plt.title('YOLOv8 Predictions\n(Pipeline 2)')
plt.axis('off')

plt.suptitle('YOLOv5 vs YOLOv8 - Side by Side Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('comparison_v5_vs_v8.png')
plt.show()
print('Saved: comparison_v5_vs_v8.png')

## Step 6 - Confidence Threshold Effects

In [ ]:
testImage = cv2.imread(sampleImages[0])

predConf025 = modelV8Best(testImage, conf=0.25)
predConf050 = modelV8Best(testImage, conf=0.50)
predConf075 = modelV8Best(testImage, conf=0.75)

annotatedConf025 = cv2.cvtColor(predConf025[0].plot(), cv2.COLOR_BGR2RGB)
annotatedConf050 = cv2.cvtColor(predConf050[0].plot(), cv2.COLOR_BGR2RGB)
annotatedConf075 = cv2.cvtColor(predConf075[0].plot(), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(annotatedConf025)
plt.title('Confidence = 0.25\n(More detections)')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(annotatedConf050)
plt.title('Confidence = 0.50\n(Default)')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(annotatedConf075)
plt.title('Confidence = 0.75\n(Fewer, more reliable)')
plt.axis('off')

plt.suptitle('YOLOv8 - Confidence Threshold Effects', fontsize=14)
plt.tight_layout()
plt.savefig('confidence_threshold_effects.png')
plt.show()
print('Saved: confidence_threshold_effects.png')

## Step 7 - Metrics and Comparison Table

In [ ]:
metricsV5 = modelV5Best.val(data=yamlPath)
metricsV8 = modelV8Best.val(data=yamlPath)

print('=' * 55)
print(f"{'Metric':<20} {'YOLOv5':>15} {'YOLOv8':>15}")
print('=' * 55)
print(f"{'mAP@0.5':<20} {round(metricsV5.box.map50, 3):>15} {round(metricsV8.box.map50, 3):>15}")
print(f"{'mAP@0.5:0.95':<20} {round(metricsV5.box.map,   3):>15} {round(metricsV8.box.map,   3):>15}")
print(f"{'Precision':<20} {round(metricsV5.box.mp,    3):>15} {round(metricsV8.box.mp,    3):>15}")
print(f"{'Recall':<20} {round(metricsV5.box.mr,    3):>15} {round(metricsV8.box.mr,    3):>15}")
print('=' * 55)

## Step 8 - Training Charts

In [ ]:
# yolov5 training chart
v5Chart = cv2.imread('runs/detect/fruit_yolov5/results.png')
v5Chart = cv2.cvtColor(v5Chart, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 6))
plt.imshow(v5Chart)
plt.title('YOLOv5 Training Results')
plt.axis('off')
plt.show()

In [ ]:
# yolov8 training chart
v8Chart = cv2.imread('runs/detect/fruit_yolov8/results.png')
v8Chart = cv2.cvtColor(v8Chart, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 6))
plt.imshow(v8Chart)
plt.title('YOLOv8 Training Results')
plt.axis('off')
plt.show()